# RAG3 - Solvency II RAG Pipeline

This notebook is a stronger template for regulatory RAG work than `RAG_pipeline2.ipynb`.

It is designed for Solvency II materials where trust, traceability, citations, and version control matter.

Core design:

1. Load authoritative files from a controlled folder.
2. Preserve metadata: source file, page, section, document type, jurisdiction, version.
3. Split by legal/regulatory structure where possible.
4. Use hybrid retrieval: BM25 keyword search + semantic vector search.
5. Optionally rerank retrieved chunks.
6. Force the LLM to answer only from cited context.
7. Include an evaluation harness for retrieval and answer quality.

Important: this notebook is an engineering template, not legal advice. Regulatory conclusions should still be reviewed by a qualified expert.

## Version améliorée par Codex

Principales corrections apportées:

- Pipeline exécutable de bout en bout: `ensure_indexes()` reconstruit Chroma si l'index est absent ou périmé.
- Embeddings et reranker plus adaptés au français/multilingue.
- Tokenisation BM25 compatible avec les accents français.
- Détection plus robuste des articles, chapitres, titres, annexes et considérants.
- Recherche hybride avec pondération BM25/vectorielle, seuil anti-résultats faibles et filtres optionnels.
- Réponse RAG avec citations obligatoires, garde-fou si aucune source utile n'est retrouvée.
- Évaluation de retrieval avec hit@k et MRR pour suivre la qualité.


In [20]:
# Install base dependencies for the BM25 pipeline.
# sentence-transformers, chromadb, and langchain-groq are optional extras.

!pip install pypdf python-docx beautifulsoup4 lxml rank-bm25 pandas pydantic

In [21]:
# Optional: create a dedicated kernel first, then add vector/LLM extras only if your environment supports them.
!/Users/coralieroland/miniconda3/envs/rag-env/bin/python -m pip install ipykernel
!/Users/coralieroland/miniconda3/envs/rag-env/bin/python -m ipykernel install --user --name rag-env --display-name "Python (rag-env)"

Installed kernelspec rag-env in /Users/coralieroland/Library/Jupyter/kernels/rag-env


In [22]:
import sys
!{sys.executable} -m pip install pypdf python-docx beautifulsoup4 lxml rank-bm25 pandas pydantic langchain-groq

In [1]:
from rank_bm25 import BM25Okapi

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import re
from datetime import datetime, timezone
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

try:
    import docx
except ImportError:
    docx = None

try:
    import chromadb
    from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
except Exception as e:
    chromadb = None
    SentenceTransformerEmbeddingFunction = None
    print(f"Chroma embedding support unavailable: {type(e).__name__}: {e}")

try:
    from sentence_transformers import CrossEncoder
except Exception as e:
    CrossEncoder = None
    print(f"CrossEncoder unavailable: {type(e).__name__}: {e}")


/Users/coralieroland/miniconda3/envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration

PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DOCS_DIR = PROJECT_DIR / "Directive"
INDEX_DIR = PROJECT_DIR / "rag3_index"
CHUNKS_PATH = INDEX_DIR / "chunks.jsonl"
MANIFEST_PATH = INDEX_DIR / "manifest.json"
AUDIT_LOG_PATH = PROJECT_DIR / "audit_log.jsonl"

# Les modèles multilingues sont plus sûrs pour les contenus réglementaires en français que les configurations par défaut limitées à l’anglais.
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
RERANKER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

TOP_K_BM25 = 30
TOP_K_VECTOR = 30
TOP_K_FINAL = 4
RRF_K = 30
BM25_RRF_WEIGHT = 0.60
VECTOR_RRF_WEIGHT = 0.40
MIN_EXTRACTED_TEXT_CHARS = 50
MIN_RETRIEVED_CHARS = 200
DOCUMENT_TYPE_BOOSTS = {
    "Directive Solvabilité II": 1.20,
    "Delegated Regulation": 1.12,
    "EIOPA material": 1.05,
    "Superviseur local": 1.08,
    "Matériel ORSA/ERSA": 1.05,
    "Matériel SFCR": 1.03,
    "Matériel RSR": 1.03,
    "Inconnu": 1.00,
}

DOCS_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

print(f"Dossier source Solvabilité II: {DOCS_DIR.resolve()}")


Dossier source Solvabilité II: /Users/coralieroland/Desktop/leRAG/Directive


## 1. Modèles de documents et de segments

Chaque passage retrouvé contient des métadonnées. C’est ce qui fait la différence entre un système RAG de démonstration et un système que l’on peut auditer.

In [4]:
# Structure les documents et les morceaux de texte pour que chaque information utilisée par le RAG
# garde son origine : fichier source, page, type de document, version, juridiction, etc.
# Cela permet de retrouver exactement d'où vient une réponse, de citer les sources,
# d'éviter les doublons et de rendre le système vérifiable.
# Définit les objets utilisés pour stocker un document complet et ses morceaux de texte.
# Chaque document ou chunk garde son contenu, sa source et ses métadonnées
# afin de savoir précisément d'où vient l'information utilisée par le RAG.
# La fonction stable_id crée un identifiant stable pour retrouver facilement
# le même document ou le même chunk à chaque exécution.
@dataclass
class LoadedDocument:
    text: str
    source_path: str
    source_name: str
    page: Optional[int] = None
    document_type: Optional[str] = None
    jurisdiction: Optional[str] = None
    version: Optional[str] = None
    temporal_note: Optional[str] = None


@dataclass
class Chunk:
    chunk_id: str
    text: str
    source_path: str
    source_name: str
    page: Optional[int]
    section: Optional[str]
    document_type: Optional[str]
    jurisdiction: Optional[str]
    version: Optional[str]
    temporal_note: Optional[str] = None


def stable_id(*parts: str) -> str:
    raw = "||".join(str(part) for part in parts)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


## 2. Metadata Rules

Ce notebook déduit des métadonnées de base à partir des noms de fichiers, mais pour un vrai travail, il faut les organiser avec soin.

Exemples de noms de fichiers recommandés :

CELEX_32009L0138_FR_TXT.pdf
EU_Directive_2009-138_FR.pdf
EIOPA_Orientations_ORSA_FR_2023.pdf
BE_BNB_Circulaire_Solvabilite-II_2024.pdf

In [5]:
# Analyse le nom d'un fichier pour déduire automatiquement ses métadonnées principales :
# type de document, juridiction, version/date et note de contexte.
# Ces informations permettent au RAG de mieux classer les sources, de distinguer les textes
# européens, locaux ou EIOPA, et de garder une trace du contexte réglementaire applicable.

def infer_metadata(path: Path) -> dict:
    name = path.stem.lower()

    if "eiopa" in name:
        document_type = "EIOPA material"
    elif "delegated" in name or "2015-35" in name:
        document_type = "Delegated Regulation"
    elif "directive" in name or "2009-138" in name or "32009l0138" in name or "celex" in name:
        document_type = "Directive Solvabilité II"
    elif "orsa" in name or "ersa" in name:
        document_type = "Matériel ORSA/ERSA"
    elif "sfcr" in name or "rapport sur la solvabilite" in name:
        document_type = "Matériel SFCR"
    elif "rsr" in name:
        document_type = "Matériel RSR"
    elif "bnb" in name or "nbb" in name:
        document_type = "Superviseur local"
    else:
        document_type = "Inconnu"

    jurisdiction = "EU" if "celex" in name or "32009l0138" in name else None
    if "bnb" in name or "nbb" in name:
        jurisdiction = "BE"
    for candidate in ["EU", "BE", "FR", "LU", "NL", "DE", "UK"]:
        if re.search(rf"(^|[_\- ]){candidate.lower()}($|[_\- ])", name):
            jurisdiction = candidate
            break

    version_match = re.search(r"(20\d{2}[-_ ]?\d{2}[-_ ]?\d{2}|v\d+(?:\.\d+)?)", name)
    version = version_match.group(1) if version_match else None

    if document_type == "Directive Solvabilité II":
        version = version or "2009/138/CE"
        temporal_note = (
            "Directive 2009/138/CE Solvabilité II; cadre applicable depuis le 01/01/2016, "
            "tel que modifié notamment par Omnibus II et sous réserve des dispositions transitoires."
        )
    elif document_type == "Delegated Regulation":
        version = version or "2015/35"
        temporal_note = (
            "Règlement délégué (UE) 2015/35 complétant Solvabilité II; "
            "à lire avec la Directive 2009/138/CE et ses modifications ultérieures."
        )
    elif document_type == "Superviseur local":
        version = version or "NBB_2016_31"
        temporal_note = (
            "Source de superviseur local belge; vérifier la date/version de la circulaire "
            "et son éventuelle consolidation avant usage prudentiel."
        )
    elif document_type == "EIOPA material":
        temporal_note = (
            "Orientation ou Q&A EIOPA; clarification de niveau 3. Vérifier la date de publication "
            "et les éventuelles mises à jour sur la page source."
        )
    else:
        temporal_note = None

    return {
        "document_type": document_type,
        "jurisdiction": jurisdiction,
        "version": version,
        "temporal_note": temporal_note,
    }


## 3. Charger les documents réglementaires

Formats pris en charge : PDF, DOCX, TXT, MD, HTML.

Pour les PDF scannés, ajoutez une étape d’OCR avant celle-ci. Ne faites pas confiance à un texte extrait vide.

In [6]:
# Charge les documents réglementaires depuis différents formats (PDF, DOCX, TXT, MD, HTML)
# et les transforme en objets LoadedDocument enrichis avec leurs métadonnées.
# Pour les PDF, le code extrait le texte page par page avec PyMuPDF, puis utilise l'OCR
# lorsque le texte extrait est presque vide, ce qui permet de traiter aussi certains PDF scannés.
# Les fichiers Word, texte, Markdown et HTML sont lus avec le chargeur adapté,
# puis tous les documents trouvés dans le dossier sont parcourus et chargés automatiquement.
def _pymupdf_ocr_text(page) -> str:
    """OCR a page with the conda Tesseract install when available."""
    if fitz is None:
        return ""

    env_bin = Path(os.environ.get("CONDA_PREFIX", Path(os.sys.executable).resolve().parent.parent)) / "bin"
    tessdata = env_bin.parent / "share" / "tessdata"
    os.environ["PATH"] = str(env_bin) + os.pathsep + os.environ.get("PATH", "")

    try:
        kwargs = {"language": "eng", "dpi": 200, "full": True}
        if tessdata.exists():
            kwargs["tessdata"] = str(tessdata)
        textpage = page.get_textpage_ocr(**kwargs)
        return page.get_text("text", textpage=textpage) or ""
    except Exception:
        return ""


def load_pdf(path: Path) -> list[LoadedDocument]:
    """Load PDF pages with PyMuPDF first, with OCR fallback for near-empty pages."""
    metadata = infer_metadata(path)
    docs = []

    if fitz is not None:
        pdf = fitz.open(str(path))
        try:
            for page_index, page in enumerate(pdf, start=1):
                text = page.get_text("text") or ""
                if len(text.strip()) < MIN_EXTRACTED_TEXT_CHARS:
                    ocr_text = _pymupdf_ocr_text(page)
                    if len(ocr_text.strip()) > len(text.strip()):
                        text = ocr_text
                if text.strip():
                    docs.append(LoadedDocument(
                        text=text,
                        source_path=str(path),
                        source_name=path.name,
                        page=page_index,
                        **metadata,
                    ))
        finally:
            pdf.close()
        return docs

    if PdfReader is None:
        raise ImportError("Install pymupdf to load PDFs robustly: pip install pymupdf")

    reader = PdfReader(str(path))
    for page_index, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(LoadedDocument(
                text=text,
                source_path=str(path),
                source_name=path.name,
                page=page_index,
                **metadata,
            ))
    return docs


def load_docx(path: Path) -> list[LoadedDocument]:
    if docx is None:
        raise ImportError("Install python-docx to load DOCX files: pip install python-docx")

    metadata = infer_metadata(path)
    document = docx.Document(str(path))
    text = "\n\n".join(p.text for p in document.paragraphs if p.text.strip())
    return [LoadedDocument(text=text, source_path=str(path), source_name=path.name, **metadata)]


def load_text_like(path: Path) -> list[LoadedDocument]:
    metadata = infer_metadata(path)
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() in {".html", ".htm"}:
        soup = BeautifulSoup(raw, "html.parser")
        raw = soup.get_text("\n", strip=True)
    return [LoadedDocument(text=raw, source_path=str(path), source_name=path.name, **metadata)]


def load_documents(folder: Path) -> list[LoadedDocument]:
    loaders = {
        ".pdf": load_pdf,
        ".docx": load_docx,
        ".txt": load_text_like,
        ".md": load_text_like,
        ".html": load_text_like,
        ".htm": load_text_like,
    }

    documents = []
    for path in sorted(folder.rglob("*")):
        if path.is_file() and path.suffix.lower() in loaders:
            loaded = loaders[path.suffix.lower()](path)
            documents.extend(loaded)
            print(f"Loaded {len(loaded):>3} item(s): {path.name}")

    return documents


##4. Découpage adapté aux textes juridiques
Ce découpeur essaie de préserver les articles, les orientations, les annexes et les paragraphes numérotés. Il applique ensuite une limite de taille afin que les segments tiennent dans le contexte du modèle.

In [7]:
SECTION_RE = re.compile(
    r"""(?imx)
    ^
    (
        article\s+\d+[a-z]*(?:\s+bis|\s+ter)?
        | considérant\s+\(?\d+\)?
        | orientation\s+\d+
        | ligne\s+directrice\s+\d+
        | guideline\s+\d+
        | chapitre\s+[ivxlcdm\d]+
        | chapter\s+[ivxlcdm\d]+
        | titre\s+[ivxlcdm\d]+
        | title\s+[ivxlcdm\d]+
        | annexe\s+[ivxlcdm\d]+
        | annex\s+[ivxlcdm\d]+
        | section\s+\d+
        | \d+\.\s+[A-ZÉÈÀÂÊÎÔÛÇ][^\n]{3,140}
    )
    """,
)

ARTICLE_SECTION_RE = re.compile(r"(?i)^article\s+\d+[a-z]*(?:\s+bis|\s+ter)?$")
ARTICLE_MAX_CHARS = 7000
DEFAULT_CHUNK_MAX_CHARS = 2400
CHUNK_OVERLAP_CHARS = 350
CHUNKING_VERSION = "article-aware-v2"


def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_by_sections_with_offsets(text: str) -> list[tuple[Optional[str], str, int]]:
    text = normalize_whitespace(text)
    matches = list(SECTION_RE.finditer(text))

    # In legal texts, numbered paragraphs such as "1." and "2." belong to the
    # surrounding article; they should not split Article 77 across pages.
    if any(ARTICLE_SECTION_RE.match(re.sub(r"\s+", " ", match.group(1).strip())) for match in matches):
        matches = [
            match for match in matches
            if not re.match(r"^\d+\.", re.sub(r"\s+", " ", match.group(1).strip()))
        ]

    if not matches:
        return [(None, text, 0)] if text else []

    sections = []
    if matches[0].start() > 0:
        prefix = text[:matches[0].start()].strip()
        if prefix:
            sections.append((None, prefix, 0))

    for index, match in enumerate(matches):
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        section_name = re.sub(r"\s+", " ", match.group(1).strip())
        section_text = text[start:end].strip()
        if section_text:
            sections.append((section_name, section_text, start))

    return sections


def split_by_sections(text: str) -> list[tuple[Optional[str], str]]:
    return [(section, section_text) for section, section_text, _ in split_by_sections_with_offsets(text)]


NUMBERED_PARAGRAPH_RE = re.compile(
    r"(?m)^(?:\d+\.|\(\d+\)|[a-z]\))\s+"
)


def split_numbered_paragraphs(text: str) -> list[str]:
    text = text.strip()
    if not text:
        return []

    matches = list(NUMBERED_PARAGRAPH_RE.finditer(text))
    if not matches:
        return [text]

    blocks = []
    if matches[0].start() > 0:
        prefix = text[:matches[0].start()].strip()
        if prefix:
            blocks.append(prefix)

    for index, match in enumerate(matches):
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        if block:
            blocks.append(block)

    return blocks


def window_text(text: str, max_chars: int = DEFAULT_CHUNK_MAX_CHARS, overlap: int = CHUNK_OVERLAP_CHARS) -> list[str]:
    text = text.strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    atomic_blocks = split_numbered_paragraphs(text)
    if len(atomic_blocks) > 1:
        windows = []
        current = ""

        for block in atomic_blocks:
            if len(block) > max_chars:
                if current:
                    windows.append(current.strip())
                    current = ""
                windows.extend(window_text(block, max_chars=max_chars, overlap=overlap))
                continue

            candidate = f"{current}\n\n{block}".strip() if current else block
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    windows.append(current.strip())
                current = block

        if current:
            windows.append(current.strip())
        return [w for w in windows if w]

    windows = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        cut = text[start:end]

        if end < len(text):
            paragraph_cut = cut.rfind("\n\n")
            sentence_cut = max(cut.rfind(". "), cut.rfind("; "), cut.rfind(": "))
            boundary = paragraph_cut if paragraph_cut > max_chars * 0.55 else sentence_cut
            if boundary > max_chars * 0.55:
                cut = cut[: boundary + 1]
                end = start + boundary + 1

        windows.append(cut.strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)

    return [w for w in windows if w]


def _group_documents_by_source(documents: list[LoadedDocument]) -> list[dict]:
    grouped = []
    by_source = {}
    for doc in documents:
        group = by_source.get(doc.source_path)
        if group is None:
            group = {
                "source_path": doc.source_path,
                "source_name": doc.source_name,
                "document_type": doc.document_type,
                "jurisdiction": doc.jurisdiction,
                "version": doc.version,
                "temporal_note": doc.temporal_note,
                "docs": [],
            }
            by_source[doc.source_path] = group
            grouped.append(group)
        group["docs"].append(doc)

    for group in grouped:
        group["docs"].sort(key=lambda doc: (doc.page is None, doc.page or 0))
    return grouped


def _combine_group_pages(group: dict) -> tuple[str, list[tuple[int, Optional[int]]]]:
    combined_parts = []
    page_offsets = []
    cursor = 0
    for doc in group["docs"]:
        text = normalize_whitespace(doc.text)
        if not text:
            continue
        if combined_parts:
            separator = "\n\n"
            combined_parts.append(separator)
            cursor += len(separator)
        page_offsets.append((cursor, doc.page))
        combined_parts.append(text)
        cursor += len(text)
    return "".join(combined_parts), page_offsets


def _page_for_offset(page_offsets: list[tuple[int, Optional[int]]], offset: int) -> Optional[int]:
    current_page = None
    for page_start, page in page_offsets:
        if page_start <= offset:
            current_page = page
        else:
            break
    return current_page


def _max_chars_for_section(section: Optional[str], section_text: str) -> int:
    if section and ARTICLE_SECTION_RE.match(section.strip()):
        return max(ARTICLE_MAX_CHARS, min(len(section_text), ARTICLE_MAX_CHARS))
    return DEFAULT_CHUNK_MAX_CHARS


def chunk_documents(documents: list[LoadedDocument]) -> list[Chunk]:
    chunks = []
    for group in _group_documents_by_source(documents):
        combined_text, page_offsets = _combine_group_pages(group)
        if not combined_text.strip():
            continue

        for section_index, (section, section_text, section_offset) in enumerate(split_by_sections_with_offsets(combined_text), start=1):
            max_chars = _max_chars_for_section(section, section_text)
            parts = window_text(section_text, max_chars=max_chars, overlap=CHUNK_OVERLAP_CHARS)
            for part_index, part in enumerate(parts, start=1):
                relative_offset = section_text.find(part[: min(len(part), 80)])
                if relative_offset < 0:
                    relative_offset = 0
                absolute_offset = section_offset + relative_offset
                page = _page_for_offset(page_offsets, absolute_offset)
                chunk_id = stable_id(
                    CHUNKING_VERSION,
                    group["source_path"],
                    str(page),
                    str(section_index),
                    str(section),
                    str(part_index),
                    part,
                )
                chunks.append(Chunk(
                    chunk_id=chunk_id,
                    text=part,
                    source_path=group["source_path"],
                    source_name=group["source_name"],
                    page=page,
                    section=section,
                    document_type=group["document_type"],
                    jurisdiction=group["jurisdiction"],
                    version=group["version"],
                    temporal_note=group["temporal_note"],
                ))
    return chunks

## 5. Construire l’index
Cette étape crée :

- `chunks.jsonl` : stockage auditable des segments
- Index vectoriel Chroma : recherche sémantique
- Index BM25 : recherche exacte de termes réglementaires

In [8]:
# Gère la création et la mise à jour de l’index documentaire du RAG.
# Le code sauvegarde les chunks dans un fichier auditable, vérifie si les sources ont changé,
# recharge les chunks existants si tout est déjà à jour, ou reconstruit les chunks si nécessaire.
# Il détecte aussi les pages presque vides pour signaler un possible besoin d’OCR.
# Ensuite, il construit l’index de recherche : un stockage JSONL pour l’audit,
# un index BM25 pour la recherche par mots-clés, et si les bibliothèques sont disponibles,
# un index vectoriel Chroma pour la recherche sémantique.

def save_chunks(chunks: list[Chunk], path: Path) -> None:
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")


def load_chunks(path: Path) -> list[Chunk]:
    with path.open("r", encoding="utf-8") as f:
        return [Chunk(**json.loads(line)) for line in f if line.strip()]


def source_manifest(folder: Path) -> dict:
    files = []
    for path in sorted(folder.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".pdf", ".docx", ".txt", ".md", ".html", ".htm"}:
            stat = path.stat()
            files.append({
                "path": str(path),
                "size": stat.st_size,
                "mtime_ns": stat.st_mtime_ns,
            })
    return {
        "embedding_model": EMBEDDING_MODEL,
        "chunking_version": CHUNKING_VERSION,
        "article_max_chars": ARTICLE_MAX_CHARS,
        "default_chunk_max_chars": DEFAULT_CHUNK_MAX_CHARS,
        "files": files,
    }


def chunks_are_current() -> bool:
    if not CHUNKS_PATH.exists() or not MANIFEST_PATH.exists():
        return False
    try:
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) == source_manifest(DOCS_DIR)
    except Exception:
        return False


def vector_index_is_ready(expected_count: Optional[int] = None) -> bool:
    if chromadb is None or SentenceTransformerEmbeddingFunction is None:
        return True

    chroma_dir = INDEX_DIR / "chroma"
    if not chroma_dir.exists():
        return False

    try:
        client = chromadb.PersistentClient(path=str(chroma_dir))
        collection = client.get_collection("solvency_ii_chunks")
        count = collection.count()
        return count > 0 and (expected_count is None or count == expected_count)
    except Exception:
        return False


def index_is_current() -> bool:
    if not chunks_are_current():
        return False
    try:
        chunk_count = sum(1 for line in CHUNKS_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
        return vector_index_is_ready(expected_count=chunk_count)
    except Exception:
        return False


def build_chunks(force: bool = False) -> list[Chunk]:
    if not force and chunks_are_current():
        print("Chunks déjà à jour.")
        return load_chunks(CHUNKS_PATH)

    documents = load_documents(DOCS_DIR)
    if not documents:
        raise ValueError(f"No supported documents found in {DOCS_DIR.resolve()}")

    empty_pages = [doc for doc in documents if len(doc.text.strip()) < MIN_EXTRACTED_TEXT_CHARS]
    if empty_pages:
        print(
            f"Attention: {len(empty_pages)} page(s)/document(s) ignorées car presque vides. "
            "OCR possiblement nécessaire."
        )
        scan_report = [
            {
                "source_name": doc.source_name,
                "source_path": doc.source_path,
                "page": doc.page,
                "extracted_chars": len(doc.text.strip()),
            }
            for doc in empty_pages
        ]
        (INDEX_DIR / "empty_extraction_report.json").write_text(
            json.dumps(scan_report, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    documents = [doc for doc in documents if len(doc.text.strip()) >= MIN_EXTRACTED_TEXT_CHARS]
    if not documents:
        raise ValueError("All documents/pages look empty after extraction. OCR is required before indexing.")

    chunks = chunk_documents(documents)
    if not chunks:
        raise ValueError("No chunks created. Check PDF extraction/OCR.")

    save_chunks(chunks, CHUNKS_PATH)
    print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

    MANIFEST_PATH.write_text(json.dumps(source_manifest(DOCS_DIR), ensure_ascii=False, indent=2), encoding="utf-8")
    return chunks


def build_indexes(force: bool = False) -> list[Chunk]:
    if not force and index_is_current():
        print("Index déjà à jour.")
        return load_chunks(CHUNKS_PATH)

    chunks = build_chunks(force=force)

    if chromadb is None or SentenceTransformerEmbeddingFunction is None:
        print("Index BM25 prêt. Index vectoriel ignoré: chromadb/sentence-transformers indisponible.")
        return chunks

    try:
        embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
        client = chromadb.PersistentClient(path=str(INDEX_DIR / "chroma"))

        try:
            client.delete_collection("solvency_ii_chunks")
        except Exception:
            pass

        collection = client.get_or_create_collection(
            name="solvency_ii_chunks",
            embedding_function=embedding_function,
            metadata={"hnsw:space": "cosine"},
        )

        batch_size = 128
        for start in range(0, len(chunks), batch_size):
            batch = chunks[start:start + batch_size]
            collection.add(
                ids=[chunk.chunk_id for chunk in batch],
                documents=[chunk.text for chunk in batch],
                metadatas=[{
                    "source_name": chunk.source_name,
                    "source_path": chunk.source_path,
                    "page": chunk.page or "",
                    "section": chunk.section or "",
                    "document_type": chunk.document_type or "",
                    "jurisdiction": chunk.jurisdiction or "",
                    "version": chunk.version or "",
                } for chunk in batch],
            )
            print(f"Indexed {min(start + batch_size, len(chunks))}/{len(chunks)} chunks")
    except Exception as e:
        print(f"Index BM25 prêt. Index vectoriel ignoré: {type(e).__name__}: {e}")
        return chunks

    print("Vector index ready")
    return chunks


def ensure_chunks() -> list[Chunk]:
    return build_chunks(force=False)


def ensure_indexes() -> list[Chunk]:
    return build_indexes(force=False)


# Run this before asking questions.
chunks = ensure_chunks()

Chunks déjà à jour.


## 6. Recherche hybride

BM25 récupère les termes exacts comme SCR, MCR, ORSA, SFCR, ajustement de volatilité, ainsi que les références aux articles.

La recherche vectorielle récupère les paraphrases et les questions conceptuelles.

Le score final utilise la fusion par rang réciproque, puis éventuellement un réordonnancement avec un cross-encoder.


In [9]:
FRENCH_STOPWORDS = {
    "le", "la", "les", "un", "une", "des", "de", "du", "d", "l", "et", "ou", "à", "a",
    "au", "aux", "en", "dans", "sur", "pour", "par", "que", "qui", "quoi", "dont",
    "est", "sont", "avec", "ce", "cet", "cette", "ces", "selon", "directive"
}

QUERY_SYNONYMS = {
    "best estimate": ["meilleure estimation"],
    "be": ["best estimate", "meilleure estimation"],
    "scr": ["capital de solvabilité requis"],
    "mcr": ["minimum de capital requis"],
    "gouvernance": [
        "système de gouvernance",
        "article 40",
        "article 41",
        "article 42",
        "article 43",
        "article 44",
        "article 45",
        "article 46",
        "article 47",
        "article 48",
        "article 49",
        "chapitre iv",
        "exigences qualitatives",
    ],
    "orsa": [
        "article 45",
        "évaluation interne des risques",
        "évaluation interne des risques et de la solvabilité",
        "ersa",
        "own risk solvency assessment",
        "own risk and solvency assessment",
    ],
    "sfcr": ["rapport sur la solvabilité et la situation financière"],
}


def expand_query(query: str) -> str:
    expanded = [query]
    lowered = query.lower()
    for key, values in QUERY_SYNONYMS.items():
        if key in lowered:
            expanded.extend(values)
    return " ".join(expanded)


def tokenize_for_bm25(text: str) -> list[str]:
    # Unicode-aware tokenization: keeps French accents and regulatory expressions such as 2009/138/CE.
    tokens = re.findall(r"[^\W_]+(?:[-/][^\W_]+)*", text.lower(), flags=re.UNICODE)
    return [token for token in tokens if token not in FRENCH_STOPWORDS and len(token) > 1]


def normalize_retrieval_scores(scores_by_id: dict[str, float]) -> dict[str, float]:
    if not scores_by_id:
        return {}
    values = list(scores_by_id.values())
    low = min(values)
    high = max(values)
    if high == low:
        return {chunk_id: 1.0 for chunk_id in scores_by_id}
    return {
        chunk_id: round((score - low) / (high - low), 3)
        for chunk_id, score in scores_by_id.items()
    }


class SolvencyRetriever:
    def __init__(self, chunks_path: Path = CHUNKS_PATH, use_reranker: bool = True):
        ensure_indexes()
        self.chunks = load_chunks(chunks_path)
        self.by_id = {chunk.chunk_id: chunk for chunk in self.chunks}
        self.bm25 = BM25Okapi([tokenize_for_bm25(chunk.text) for chunk in self.chunks])
        self.last_scores: dict[str, float] = {}
        self.last_raw_scores: dict[str, float] = {}

        self.collection = None
        if chromadb is None or SentenceTransformerEmbeddingFunction is None:
            print("Recherche vectorielle ignorée: chromadb/sentence-transformers indisponible.")
        else:
            try:
                embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
                client = chromadb.PersistentClient(path=str(INDEX_DIR / "chroma"))
                self.collection = client.get_or_create_collection(
                    name="solvency_ii_chunks",
                    embedding_function=embedding_function,
                )
            except Exception as e:
                print(f"Recherche vectorielle ignorée: {type(e).__name__}: {e}")

        self.reranker = None
        if use_reranker and CrossEncoder is not None:
            try:
                self.reranker = CrossEncoder(RERANKER_MODEL)
            except Exception as e:
                print(f"Reranker ignoré: {type(e).__name__}: {e}")

    def bm25_search(self, query: str, k: int = TOP_K_BM25) -> list[str]:
        scores = self.bm25.get_scores(tokenize_for_bm25(expand_query(query)))
        ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
        return [self.chunks[index].chunk_id for index, score in ranked if score > 0]

    def vector_search(self, query: str, k: int = TOP_K_VECTOR) -> list[str]:
        if self.collection is None:
            return []
        result = self.collection.query(query_texts=[expand_query(query)], n_results=k)
        return [chunk_id for chunk_id in result.get("ids", [[]])[0] if chunk_id in self.by_id]

    def retrieve(
        self,
        query: str,
        k: int = TOP_K_FINAL,
        document_type: Optional[str] = None,
        jurisdiction: Optional[str] = None,
    ) -> list[Chunk]:
        bm25_ids = self.bm25_search(query)
        vector_ids = self.vector_search(query)

        fused = {}
        for rank, chunk_id in enumerate(bm25_ids, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + BM25_RRF_WEIGHT / (RRF_K + rank)
        for rank, chunk_id in enumerate(vector_ids, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + VECTOR_RRF_WEIGHT / (RRF_K + rank)

        boosted = {}
        for chunk_id, score in fused.items():
            chunk = self.by_id[chunk_id]
            boost = DOCUMENT_TYPE_BOOSTS.get(chunk.document_type or "Inconnu", 1.0)
            boosted[chunk_id] = score * boost

        candidates = [
            self.by_id[chunk_id]
            for chunk_id, score in sorted(boosted.items(), key=lambda item: item[1], reverse=True)
        ]
        self.last_raw_scores = dict(boosted)
        self.last_scores = normalize_retrieval_scores(self.last_raw_scores)

        if document_type:
            candidates = [c for c in candidates if c.document_type == document_type]
        if jurisdiction:
            candidates = [c for c in candidates if c.jurisdiction == jurisdiction]

        if self.reranker is not None and candidates:
            pairs = [(query, chunk.text) for chunk in candidates]
            scores = self.reranker.predict(pairs)
            ranked_pairs = sorted(zip(scores, candidates), key=lambda item: item[0], reverse=True)
            candidates = [chunk for score, chunk in ranked_pairs]
            self.last_raw_scores = {chunk.chunk_id: float(score) for score, chunk in ranked_pairs}
            self.last_scores = normalize_retrieval_scores(self.last_raw_scores)

        returned = candidates[:k]
        self.last_scores = {chunk.chunk_id: self.last_scores.get(chunk.chunk_id, 0.0) for chunk in returned}
        return returned


## 7. Cited Answer Generation

The answer prompt is deliberately strict. If the retrieved context does not support the answer, the model must say so. Cette partie combine une recherche par mots exacts et une recherche par sens afin de retrouver les passages les plus pertinents dans les documents réglementaires.


In [10]:
def format_citation(chunk: Chunk, number: int) -> str:
    page = f", page {chunk.page}" if chunk.page else ""
    section = f", {chunk.section}" if chunk.section else ""
    version = f", version/date {chunk.version}" if chunk.version else ""
    temporal = f", temporalité: {chunk.temporal_note}" if getattr(chunk, "temporal_note", None) else ""
    return f"[{number}] {chunk.source_name}{page}{section}{version}{temporal}"


def format_context(chunks: list[Chunk]) -> str:
    blocks = []
    for index, chunk in enumerate(chunks, start=1):
        citation = format_citation(chunk, index)
        temporal = f"\nTEMPORALITÉ: {chunk.temporal_note}" if getattr(chunk, "temporal_note", None) else ""
        blocks.append(f"SOURCE {index}: {citation}{temporal}\n{chunk.text}")
    return "\n\n---\n\n".join(blocks)


SYSTEM_PROMPT = """
Tu es un expert Solvabilité II qui explique la réglementation à des actuaires 
et professionnels de l'assurance qualifiés. Tu combines deux qualités :
la rigueur d'un juriste et la pédagogie d'un consultant senior.

Pour chaque réponse, suis EXACTEMENT cette structure en 4 blocs :

---

## 💡 En clair
Explique le concept en 2-3 phrases simples, comme si tu l'expliquais 
à un collègue intelligent qui découvre le sujet. Pas de jargon inutile. 
Utilise des analogies si elles aident.

## 📐 Ce que dit la réglementation
Donne la définition formelle et technique. Cite les articles exacts 
entre crochets [1], [2]. Mentionne explicitement si c'est la Directive, 
les Actes Délégués, ou les Guidelines EIOPA — la distinction compte. Avec la rigueur pour un actuaire.

## ⚙️ Comment ça marche en pratique
Donne un exemple concret ou une application pratique pour un assureur vie 
ou non-vie belge et français. Si une formule existe dans les sources, cite-la ici.

## ⚠️ Limites et points d'attention
Ce que les sources ne couvrent pas. Ce qu'un superviseur regarderait 
en priorité. Ce qui nécessite un avis expert complémentaire.

---

Règles absolues :
- N'invente aucun chiffre, seuil ou formule absent des sources.
- Si les sources sont insuffisantes sur un point, dis-le explicitement 
  dans "Limites" plutôt que d'improviser.
- La section "En clair" ne cite pas d'articles — elle explique.
- La section "Réglementation" cite toujours ses sources.
- Réponds en français.
""".strip()


CITATION_RULES = """
Quand tu cites une source, utilise ce format précis :
- Directive : "selon l'Article 77 §2 de la Directive [1]"
- Actes Délégués : "les Actes Délégués précisent à l'Article 17 [2]"
- EIOPA Guidelines : "les Orientations EIOPA recommandent [3]"
- BNB/ACPR : "la circulaire nationale exige [4]"

Ne dis jamais juste "selon [1]" sans nommer le texte.
La hiérarchie des normes compte :
Directive > Actes Délégués > Guidelines EIOPA > Circulaires nationales.
""".strip()


def _chunk_audit_payload(chunk: Chunk, score: Optional[float] = None) -> dict:
    return {
        "chunk_id": chunk.chunk_id,
        "source_name": chunk.source_name,
        "source_path": chunk.source_path,
        "page": chunk.page,
        "section": chunk.section,
        "document_type": chunk.document_type,
        "jurisdiction": chunk.jurisdiction,
        "version": chunk.version,
        "temporal_note": getattr(chunk, "temporal_note", None),
        "retrieval_score": score,
        "text_preview": normalize_whitespace(chunk.text)[:500],
    }


def audit_log_query(
    question: str,
    chunks: list[Chunk],
    retrieval_scores: Optional[dict[str, float]] = None,
    mode: str = "hybrid",
    use_llm: bool = True,
    model: str = "llama-3.3-70b-versatile",
    answer: str = "",
) -> None:
    retrieval_scores = retrieval_scores or {}
    event = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "user": os.environ.get("USER") or os.environ.get("USERNAME") or "unknown",
        "question": question,
        "mode": mode,
        "llm_requested": use_llm,
        "model": model if use_llm else None,
        "answer_preview": normalize_whitespace(answer)[:1000],
        "sources": [
            _chunk_audit_payload(chunk, retrieval_scores.get(chunk.chunk_id))
            for chunk in chunks
        ],
    }
    AUDIT_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with AUDIT_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")


def answer_without_llm(chunks: list[Chunk]) -> str:
    if not chunks or sum(len(chunk.text) for chunk in chunks) < MIN_RETRIEVED_CHARS:
        return "Je ne sais pas à partir des sources fournies."

    excerpts = []
    for index, chunk in enumerate(chunks[:3], start=1):
        snippet = normalize_whitespace(chunk.text).replace("\n", " ")
        if len(snippet) > 320:
            snippet = snippet[:317].rsplit(" ", 1)[0] + "..."
        excerpts.append(f"[{index}] {snippet}")

    return "LLM indisponible pour cette session. Voici les extraits les plus pertinents :\n\n" + "\n\n".join(excerpts)


def ensure_groq_api_key(prompt_if_missing: bool = False) -> bool:
    if os.environ.get("GROQ_API_KEY"):
        return True

    if prompt_if_missing:
        from getpass import getpass
        value = getpass("Enter GROQ_API_KEY: ").strip()
        if value:
            os.environ["GROQ_API_KEY"] = value

    return bool(os.environ.get("GROQ_API_KEY"))


def build_messages(
    question: str,
    chunks: list[Chunk],
    history: Optional[list[dict]] = None,
    extra_instruction: str = "",
) -> list[dict]:
    messages = [
        {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}\n\n{CITATION_RULES}",
        }
    ]

    previous_questions = [
        normalize_whitespace(str(item.get("question", "")))
        for item in (history or [])[-3:]
        if normalize_whitespace(str(item.get("question", "")))
    ]

    history_hint = ""
    if previous_questions:
        history_hint = "Questions précédentes, pour contexte uniquement :\n" + "\n".join(
            f"- {item}" for item in previous_questions
        ) + "\n\n"

    extra_text = ""
    if extra_instruction.strip():
        extra_text = extra_instruction.strip() + "\n\n"

    messages.append(
        {
            "role": "user",
            "content": (
                "LANGUE OBLIGATOIRE : réponds exclusivement en français. "
                "Ne réponds jamais en anglais. "
                "Ta réponse doit commencer exactement par `## 💡 En clair`, "
                "puis suivre les 4 blocs demandés dans le prompt système.\n\n"
                f"{history_hint}"
                f"{extra_text}"
                f"Question : {question}\n\nSources :\n{format_context(chunks)}"
            ),
        }
    )

    return messages


def answer_with_groq(
    question: str,
    chunks: list[Chunk],
    model: str = "llama-3.3-70b-versatile",
    history: Optional[list[dict]] = None,
    extra_instruction: str = "",
) -> str:
    if not chunks or sum(len(chunk.text) for chunk in chunks) < MIN_RETRIEVED_CHARS:
        return "Je ne sais pas à partir des sources fournies."

    if not ensure_groq_api_key(prompt_if_missing=False):
        return answer_without_llm(chunks)

    try:
        from langchain_groq import ChatGroq
    except Exception as e:
        print(f"LLM Groq indisponible: {type(e).__name__}: {e}")
        return answer_without_llm(chunks)

    try:
        llm = ChatGroq(model=model, temperature=0)
        response = llm.invoke(
            build_messages(
                question,
                chunks,
                history=history,
                extra_instruction=extra_instruction,
            )
        )
        return response.content
    except Exception as e:
        error = f"Appel Groq impossible: {type(e).__name__}: {e}"
        print(error)

        fallback = answer_without_llm(chunks)
        if "Voici les extraits les plus pertinents :" in fallback:
            fallback = fallback.split(
                "Voici les extraits les plus pertinents :",
                1,
            )[-1].strip()

        return error + "\n\nVoici les extraits les plus pertinents :\n\n" + fallback


def ask(
    question: str,
    use_llm: bool = True,
    use_reranker: bool = True,
    audience: str = "expert",
    history: Optional[list[dict]] = None,
) -> dict:
    retriever = SolvencyRetriever(use_reranker=use_reranker)
    chunks = retriever.retrieve(question)

    print("Sources retrouvées :")
    for index, chunk in enumerate(chunks, start=1):
        print(format_citation(chunk, index))

    if audience == "vulgarise":
        extra = """
Important : l'utilisateur veut comprendre le concept,
pas mémoriser les articles. Privilégie les analogies
et les exemples concrets. Garde les citations mais
mets-les entre parenthèses en fin de phrase, pas en avant.
""".strip()
    else:
        extra = ""

    model = "llama-3.3-70b-versatile"

    if use_llm:
        answer = answer_with_groq(
            question,
            chunks,
            model=model,
            extra_instruction=extra,
            history=history,
        )
    else:
        answer = "LLM désactivé. Inspecte les sources retrouvées ci-dessus."

    audit_log_query(
        question=question,
        chunks=chunks,
        retrieval_scores=retriever.last_scores,
        mode="hybrid",
        use_llm=use_llm,
        model=model,
        answer=answer,
    )

    return {
        "question": question,
        "answer": answer,
        "chunks": chunks,
        "retrieval_scores": retriever.last_scores,
    }


In [11]:
class SolvencyBM25Retriever:
    def __init__(self, chunks_path: Path = CHUNKS_PATH):
        ensure_chunks()
        self.chunks = load_chunks(chunks_path)
        self.bm25 = BM25Okapi([tokenize_for_bm25(chunk.text) for chunk in self.chunks])
        self.last_scores: dict[str, float] = {}
        self.last_raw_scores: dict[str, float] = {}

    def retrieve(self, query: str, k: int = TOP_K_FINAL) -> list[Chunk]:
        scores = self.bm25.get_scores(tokenize_for_bm25(expand_query(query)))
        ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)
        scored_results = [
            (self.chunks[index], float(score))
            for index, score in ranked
            if score > 0
        ][:k]
        self.last_raw_scores = {chunk.chunk_id: score for chunk, score in scored_results}
        self.last_scores = normalize_retrieval_scores(self.last_raw_scores)
        return [chunk for chunk, score in scored_results]


In [12]:
def ask_bm25(
    question: str,
    use_llm: bool = True,
    audience: str = "expert",
    history: Optional[list[dict]] = None,
) -> dict:
    retriever = SolvencyBM25Retriever()
    chunks = retriever.retrieve(question)

    print("Sources retrouvées :")
    for index, chunk in enumerate(chunks, start=1):
        print(format_citation(chunk, index))

    if audience == "vulgarise":
        extra = """
        Important : l'utilisateur veut comprendre le concept,
        pas mémoriser les articles. Privilégie les analogies
        et les exemples concrets. Garde les citations mais
        mets-les entre parenthèses en fin de phrase, pas en avant.
        """
    else:
        extra = ""

    model = "llama-3.3-70b-versatile"
    if use_llm:
        answer = answer_with_groq(
            question,
            chunks,
            model=model,
            extra_instruction=extra,
            history=history,
        )
    else:
        answer = "LLM désactivé. Inspecte les sources retrouvées ci-dessus."

    audit_log_query(
        question=question,
        chunks=chunks,
        retrieval_scores=retriever.last_scores,
        mode="bm25",
        use_llm=use_llm,
        model=model,
        answer=answer,
    )

    return {
        "question": question,
        "answer": answer,
        "chunks": chunks,
        "retrieval_scores": retriever.last_scores,
    }


## 8. Banc d’évaluation

Trust comes from testing. Créez un petit jeu de données de référence avec des questions et les noms de sources ou sections attendus.

20 à 50 questions couvrant le SCR, le MCR, l’ORSA, le SFCR, le RSR, les provisions techniques, la gouvernance, l’externalisation et les attentes du superviseur local.

In [13]:
EVAL_QUESTIONS = [
    # Pilier 1 - Provisions techniques
    {
        "question": "Que dit l'Article 77 sur le Best Estimate ?",
        "expected_source_contains": ["meilleure estimation", "flux de trésorerie"],
    },
    {
        "question": "Comment se calcule la Risk Margin ?",
        "expected_source_contains": ["marge de risque", "coût du capital"],
    },
    {
        "question": "Qu'est-ce que la segmentation des engagements selon l'Article 80 ?",
        "expected_source_contains": ["segmentation", "ligne d'activité"],
    },
    {
        "question": "Quelles hypothèses pour les flux de trésorerie futurs ?",
        "expected_source_contains": ["projection", "flux"],
    },

    # Pilier 1 - Capital
    {
        "question": "Définition du SCR selon l'Article 101 ?",
        "expected_source_contains": ["capital de solvabilité requis", "article 101"],
    },
    {
        "question": "Comment se calcule le MCR ?",
        "expected_source_contains": ["minimum de capital", "mcr"],
    },
    {
        "question": "Quels sont les modules de risque de la formule standard ?",
        "expected_source_contains": ["module", "formule standard"],
    },
    {
        "question": "Qu'est-ce que le SCR opérationnel ?",
        "expected_source_contains": ["risque opérationnel"],
    },

    # Pilier 2 - Gouvernance
    {
        "question": "Quelles sont les exigences du système de gouvernance Article 41 ?",
        "expected_source_contains": ["gouvernance", "article 41"],
    },
    {
        "question": "Que prévoit l'Article 45 sur l'ORSA ?",
        "expected_source_contains": ["orsa", "article 45"],
    },
    {
        "question": "Quelles sont les quatre fonctions clés sous SII ?",
        "expected_source_contains": ["fonction", "actuarielle", "audit", "gestion des risques"],
    },
    {
        "question": "Exigences fit and proper Article 42 ?",
        "expected_source_contains": ["compétence", "honorabilité"],
    },

    # Pilier 3 - Reporting
    {
        "question": "Que doit contenir le SFCR ?",
        "expected_source_contains": ["sfcr", "rapport"],
    },
    {
        "question": "Qu'est-ce que le RSR ?",
        "expected_source_contains": ["rsr", "superviseur"],
    },
    {
        "question": "Quels QRTs sont obligatoires ?",
        "expected_source_contains": ["qrt", "états quantitatifs"],
    },

    # Groupes
    {
        "question": "Comment se calcule le SCR groupe ?",
        "expected_source_contains": ["groupe", "consolidé"],
    },
    {
        "question": "Qu'est-ce que la diversification au niveau groupe ?",
        "expected_source_contains": ["diversification", "groupe"],
    },

    # Investissements
    {
        "question": "Principe de la personne prudente Article 132 ?",
        "expected_source_contains": ["personne prudente", "article 132"],
    },

    # Réassurance
    {
        "question": "Comment la réassurance réduit-elle le SCR ?",
        "expected_source_contains": ["réassurance", "atténuation"],
    },

    # Modèle interne
    {
        "question": "Conditions d'approbation d'un modèle interne ?",
        "expected_source_contains": ["modèle interne", "approbation"],
    },
]


def evaluate_retrieval(eval_questions: list[dict] = EVAL_QUESTIONS, k: int = TOP_K_FINAL) -> pd.DataFrame:
    retriever = SolvencyRetriever(use_reranker=False)
    rows = []

    for item in eval_questions:
        question = item["question"]
        expected_terms = [term.lower() for term in item.get("expected_source_contains", [])]
        chunks = retriever.retrieve(question, k=k)

        first_hit_rank = None
        for rank, chunk in enumerate(chunks, start=1):
            blob = f"{chunk.source_name} {chunk.section or ''} {chunk.text}".lower()
            matched_terms = [term for term in expected_terms if term in blob]
            coverage = len(matched_terms) / len(expected_terms) if expected_terms else 1.0
            if coverage >= 0.5:
                first_hit_rank = rank
                break

        rows.append({
            "question": question,
            "expected_terms": expected_terms,
            "match_rule": "term coverage >= 50%",
            f"hit@{k}": first_hit_rank is not None,
            "mrr": 0 if first_hit_rank is None else round(1 / first_hit_rank, 3),
            "first_hit_rank": first_hit_rank,
            "top_sources": "; ".join(format_citation(chunk, i + 1) for i, chunk in enumerate(chunks[:3])),
        })

    return pd.DataFrame(rows)


# Run manually in notebook when needed:
#evaluation = evaluate_retrieval()
#evaluation


In [15]:
# ============================================================
# Retrieval Evaluation Questions
# Solvency II RAG MVP
# ============================================================

import re
import unicodedata
import pandas as pd
EVAL_QUESTIONS = [
    # Pilier 1 - Provisions techniques
    {"category": "Pilier 1 - Provisions techniques", "question": "Que dit l'Article 77 sur le Best Estimate ?", "expected_source_contains": ["meilleure estimation", "flux de trésorerie"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Comment se calcule la Risk Margin ?", "expected_source_contains": ["marge de risque", "coût du capital"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Qu'est-ce que la segmentation des engagements selon l'Article 80 ?", "expected_source_contains": ["segmentation", "ligne d'activité"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Quelles hypothèses doivent être utilisées pour projeter les flux de trésorerie futurs ?", "expected_source_contains": ["projection", "flux de trésorerie"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Quelle est la différence entre Best Estimate et provisions techniques ?", "expected_source_contains": ["meilleure estimation", "provisions techniques"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Que signifie une valorisation market-consistent des provisions techniques ?", "expected_source_contains": ["marché", "provisions techniques"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Quels flux doivent être pris en compte dans le calcul du Best Estimate ?", "expected_source_contains": ["flux de trésorerie", "prestations", "frais"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Comment les options et garanties financières sont-elles prises en compte dans les provisions techniques ?", "expected_source_contains": ["options", "garanties", "provisions techniques"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Comment sont traitées les frontières des contrats dans Solvabilité II ?", "expected_source_contains": ["frontières des contrats", "contrat"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Quelles exigences s'appliquent à la qualité des données pour le calcul des provisions techniques ?", "expected_source_contains": ["qualité des données", "provisions techniques"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Comment les frais futurs sont-ils pris en compte dans le Best Estimate ?", "expected_source_contains": ["frais", "flux de trésorerie"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Comment les participations bénéficiaires futures sont-elles intégrées dans les provisions techniques ?", "expected_source_contains": ["participation aux bénéfices", "provisions techniques"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Que signifie l'actualisation des flux futurs dans le calcul du Best Estimate ?", "expected_source_contains": ["actualisation", "flux de trésorerie"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Quelle courbe de taux doit être utilisée pour actualiser les provisions techniques ?", "expected_source_contains": ["courbe des taux", "actualisation"]},
    {"category": "Pilier 1 - Provisions techniques", "question": "Qu'est-ce que le volatility adjustment dans Solvabilité II ?", "expected_source_contains": ["volatility adjustment", "ajustement de volatilité"]},

    # Pilier 1 - Capital
    {"category": "Pilier 1 - Capital", "question": "Définition du SCR selon l'Article 101 ?", "expected_source_contains": ["capital de solvabilité requis", "article 101"]},
    {"category": "Pilier 1 - Capital", "question": "Comment se calcule le MCR ?", "expected_source_contains": ["minimum de capital", "mcr"]},
    {"category": "Pilier 1 - Capital", "question": "Quels sont les modules de risque de la formule standard ?", "expected_source_contains": ["module", "formule standard"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le SCR opérationnel ?", "expected_source_contains": ["risque opérationnel"]},
    {"category": "Pilier 1 - Capital", "question": "Quelle est la différence entre SCR et MCR ?", "expected_source_contains": ["capital de solvabilité requis", "minimum de capital"]},
    {"category": "Pilier 1 - Capital", "question": "Quel est l'horizon temporel utilisé pour calibrer le SCR ?", "expected_source_contains": ["un an", "99,5"]},
    {"category": "Pilier 1 - Capital", "question": "Que signifie une Value-at-Risk à 99,5% dans Solvabilité II ?", "expected_source_contains": ["value-at-risk", "99,5"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le BSCR ?", "expected_source_contains": ["capital de solvabilité requis de base", "bscr"]},
    {"category": "Pilier 1 - Capital", "question": "Comment fonctionne la diversification entre modules de risque ?", "expected_source_contains": ["diversification", "corrélation"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le risque de marché dans la formule standard ?", "expected_source_contains": ["risque de marché", "formule standard"]},
    {"category": "Pilier 1 - Capital", "question": "Quels sous-modules composent le risque de marché ?", "expected_source_contains": ["taux d'intérêt", "actions", "spread"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le risque de souscription vie ?", "expected_source_contains": ["souscription vie", "mortalité", "longévité"]},
    {"category": "Pilier 1 - Capital", "question": "Quels sont les sous-modules du risque de souscription vie ?", "expected_source_contains": ["mortalité", "longévité", "rachat"]},
    {"category": "Pilier 1 - Capital", "question": "Comment le risque de frais est-il traité dans le SCR vie ?", "expected_source_contains": ["frais", "souscription vie"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le risque de rachat dans Solvabilité II ?", "expected_source_contains": ["rachat", "souscription vie"]},
    {"category": "Pilier 1 - Capital", "question": "Comment le risque de mortalité est-il choqué dans la formule standard ?", "expected_source_contains": ["mortalité", "choc"]},
    {"category": "Pilier 1 - Capital", "question": "Comment le risque de longévité est-il choqué dans la formule standard ?", "expected_source_contains": ["longévité", "choc"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que le risque de catastrophe vie ?", "expected_source_contains": ["catastrophe", "vie"]},
    {"category": "Pilier 1 - Capital", "question": "Comment les impôts différés peuvent-ils réduire le SCR ?", "expected_source_contains": ["impôts différés", "absorption des pertes"]},
    {"category": "Pilier 1 - Capital", "question": "Qu'est-ce que la capacité d'absorption des pertes des provisions techniques ?", "expected_source_contains": ["absorption des pertes", "provisions techniques"]},

    # Pilier 1 - Fonds propres
    {"category": "Pilier 1 - Fonds propres", "question": "Que sont les fonds propres éligibles sous Solvabilité II ?", "expected_source_contains": ["fonds propres", "éligibles"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Quelle est la différence entre fonds propres de base et fonds propres auxiliaires ?", "expected_source_contains": ["fonds propres de base", "fonds propres auxiliaires"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Comment les fonds propres sont-ils classés en tiers ?", "expected_source_contains": ["tier", "fonds propres"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Quelles sont les caractéristiques des fonds propres Tier 1 ?", "expected_source_contains": ["tier 1", "fonds propres"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Quelles limites s'appliquent à l'éligibilité des fonds propres pour couvrir le SCR ?", "expected_source_contains": ["éligibilité", "scr", "fonds propres"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Quelles limites s'appliquent à l'éligibilité des fonds propres pour couvrir le MCR ?", "expected_source_contains": ["éligibilité", "mcr", "fonds propres"]},
    {"category": "Pilier 1 - Fonds propres", "question": "Qu'est-ce que l'excédent d'actif sur passif dans le bilan économique Solvabilité II ?", "expected_source_contains": ["actif", "passif", "fonds propres"]},

    # Pilier 2 - Gouvernance et ORSA
    {"category": "Pilier 2 - Gouvernance", "question": "Quelles sont les exigences du système de gouvernance Article 41 ?", "expected_source_contains": ["gouvernance", "article 41"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Que prévoit l'Article 45 sur l'ORSA ?", "expected_source_contains": ["orsa", "article 45"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quelles sont les quatre fonctions clés sous Solvabilité II ?", "expected_source_contains": ["fonction", "actuarielle", "audit", "gestion des risques"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Exigences fit and proper Article 42 ?", "expected_source_contains": ["compétence", "honorabilité"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quel est le rôle de la fonction actuarielle sous Solvabilité II ?", "expected_source_contains": ["fonction actuarielle", "provisions techniques"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quel est le rôle de la fonction gestion des risques ?", "expected_source_contains": ["gestion des risques", "système"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quel est le rôle de la fonction conformité ?", "expected_source_contains": ["conformité", "fonction"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quel est le rôle de la fonction audit interne ?", "expected_source_contains": ["audit interne", "fonction"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Que doit contenir une politique de gestion des risques ?", "expected_source_contains": ["politique", "gestion des risques"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quelles exigences s'appliquent au contrôle interne ?", "expected_source_contains": ["contrôle interne", "gouvernance"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Quelles règles s'appliquent à l'externalisation sous Solvabilité II ?", "expected_source_contains": ["externalisation", "fonction"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Que signifie le principe de proportionnalité dans la gouvernance Solvabilité II ?", "expected_source_contains": ["proportionnalité", "gouvernance"]},
    {"category": "Pilier 2 - ORSA", "question": "Quels éléments doivent être pris en compte dans l'ORSA ?", "expected_source_contains": ["orsa", "besoin global de solvabilité"]},
    {"category": "Pilier 2 - ORSA", "question": "Quelle est la fréquence de réalisation de l'ORSA ?", "expected_source_contains": ["orsa", "régulièrement"]},
    {"category": "Pilier 2 - ORSA", "question": "Comment l'ORSA est-il lié à la stratégie de l'entreprise ?", "expected_source_contains": ["orsa", "stratégie"]},
    {"category": "Pilier 2 - ORSA", "question": "Qu'est-ce que l'appétence au risque ?", "expected_source_contains": ["appétence au risque", "risque"]},
    {"category": "Pilier 2 - Gouvernance", "question": "Comment le conseil d'administration intervient-il dans le système de gouvernance ?", "expected_source_contains": ["administration", "gouvernance"]},

    # Pilier 3 - Reporting et transparence
    {"category": "Pilier 3 - Reporting", "question": "Que doit contenir le SFCR ?", "expected_source_contains": ["sfcr", "rapport"]},
    {"category": "Pilier 3 - Reporting", "question": "Qu'est-ce que le RSR ?", "expected_source_contains": ["rsr", "superviseur"]},
    {"category": "Pilier 3 - Reporting", "question": "Quels QRTs sont obligatoires ?", "expected_source_contains": ["qrt", "états quantitatifs"]},
    {"category": "Pilier 3 - Reporting", "question": "Quelle est la différence entre SFCR et RSR ?", "expected_source_contains": ["sfcr", "rsr"]},
    {"category": "Pilier 3 - Reporting", "question": "À qui est destiné le SFCR ?", "expected_source_contains": ["public", "sfcr"]},
    {"category": "Pilier 3 - Reporting", "question": "À qui est destiné le RSR ?", "expected_source_contains": ["superviseur", "rsr"]},
    {"category": "Pilier 3 - Reporting", "question": "Quelles informations sur les fonds propres doivent être publiées dans le SFCR ?", "expected_source_contains": ["fonds propres", "sfcr"]},
    {"category": "Pilier 3 - Reporting", "question": "Quelles informations sur le SCR doivent être publiées dans le SFCR ?", "expected_source_contains": ["scr", "sfcr"]},
    {"category": "Pilier 3 - Reporting", "question": "Quelles informations sur les provisions techniques doivent être publiées dans le SFCR ?", "expected_source_contains": ["provisions techniques", "sfcr"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.02.01 ?", "expected_source_contains": ["bilan", "s.02.01"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.05.01 ?", "expected_source_contains": ["primes", "sinistres", "s.05.01"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.12.01 en assurance vie ?", "expected_source_contains": ["provisions techniques", "vie", "s.12.01"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.23.01 ?", "expected_source_contains": ["fonds propres", "s.23.01"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.25.01 ?", "expected_source_contains": ["scr", "formule standard", "s.25.01"]},
    {"category": "Pilier 3 - QRT", "question": "Que contient le QRT S.28.01 ou S.28.02 relatif au MCR ?", "expected_source_contains": ["mcr", "minimum de capital"]},

    # Groupes
    {"category": "Groupes", "question": "Comment se calcule le SCR groupe ?", "expected_source_contains": ["groupe", "consolidé"]},
    {"category": "Groupes", "question": "Qu'est-ce que la diversification au niveau groupe ?", "expected_source_contains": ["diversification", "groupe"]},
    {"category": "Groupes", "question": "Qu'est-ce que le contrôle de groupe sous Solvabilité II ?", "expected_source_contains": ["contrôle de groupe", "groupe"]},
    {"category": "Groupes", "question": "Comment sont traitées les transactions intra-groupe ?", "expected_source_contains": ["intra-groupe", "transactions"]},
    {"category": "Groupes", "question": "Qu'est-ce que la solvabilité de groupe ?", "expected_source_contains": ["solvabilité de groupe", "fonds propres"]},
    {"category": "Groupes", "question": "Quelle est la différence entre méthode de consolidation et méthode déduction-agrégation ?", "expected_source_contains": ["consolidation", "déduction", "agrégation"]},

    # Investissements
    {"category": "Investissements", "question": "Principe de la personne prudente Article 132 ?", "expected_source_contains": ["personne prudente", "article 132"]},
    {"category": "Investissements", "question": "Quelles exigences s'appliquent aux investissements sous Solvabilité II ?", "expected_source_contains": ["investissements", "personne prudente"]},
    {"category": "Investissements", "question": "Comment une entreprise d'assurance doit-elle gérer le risque de concentration des actifs ?", "expected_source_contains": ["concentration", "investissements"]},
    {"category": "Investissements", "question": "Comment le risque de spread est-il pris en compte dans le SCR marché ?", "expected_source_contains": ["spread", "risque de marché"]},
    {"category": "Investissements", "question": "Comment le risque actions est-il pris en compte dans le SCR marché ?", "expected_source_contains": ["actions", "risque de marché"]},
    {"category": "Investissements", "question": "Comment le risque immobilier est-il pris en compte dans le SCR marché ?", "expected_source_contains": ["immobilier", "risque de marché"]},
    {"category": "Investissements", "question": "Comment le risque de taux d'intérêt est-il pris en compte dans le SCR marché ?", "expected_source_contains": ["taux d'intérêt", "risque de marché"]},
    {"category": "Investissements", "question": "Qu'est-ce que l'ALM dans le contexte Solvabilité II ?", "expected_source_contains": ["gestion actif-passif", "alm"]},

    # Réassurance
    {"category": "Réassurance", "question": "Comment la réassurance réduit-elle le SCR ?", "expected_source_contains": ["réassurance", "atténuation"]},
    {"category": "Réassurance", "question": "Quelles conditions doivent être respectées pour reconnaître une technique d'atténuation du risque ?", "expected_source_contains": ["atténuation du risque", "technique"]},
    {"category": "Réassurance", "question": "Comment le risque de contrepartie lié à la réassurance est-il pris en compte ?", "expected_source_contains": ["contrepartie", "réassurance"]},
    {"category": "Réassurance", "question": "Qu'est-ce qu'un effet d'atténuation du risque dans la formule standard ?", "expected_source_contains": ["atténuation", "risque"]},
    {"category": "Réassurance", "question": "Quels risques résiduels peuvent subsister après réassurance ?", "expected_source_contains": ["réassurance", "risque"]},

    # Modèle interne
    {"category": "Modèle interne", "question": "Conditions d'approbation d'un modèle interne ?", "expected_source_contains": ["modèle interne", "approbation"]},
    {"category": "Modèle interne", "question": "Quelle est la différence entre formule standard et modèle interne ?", "expected_source_contains": ["formule standard", "modèle interne"]},
    {"category": "Modèle interne", "question": "Qu'est-ce qu'un modèle interne partiel ?", "expected_source_contains": ["modèle interne partiel", "modèle interne"]},
    {"category": "Modèle interne", "question": "Qu'est-ce que le use test dans le cadre d'un modèle interne ?", "expected_source_contains": ["use test", "modèle interne"]},
    {"category": "Modèle interne", "question": "Quelles exigences de validation s'appliquent à un modèle interne ?", "expected_source_contains": ["validation", "modèle interne"]},
    {"category": "Modèle interne", "question": "Quelles exigences de documentation s'appliquent à un modèle interne ?", "expected_source_contains": ["documentation", "modèle interne"]},
    {"category": "Modèle interne", "question": "Qu'est-ce que le test de calibration d'un modèle interne ?", "expected_source_contains": ["calibration", "modèle interne"]},
    {"category": "Modèle interne", "question": "Qu'est-ce que le test statistique d'un modèle interne ?", "expected_source_contains": ["statistique", "modèle interne"]},
    {"category": "Modèle interne", "question": "Comment le superviseur approuve-t-il un modèle interne ?", "expected_source_contains": ["superviseur", "approbation", "modèle interne"]},

    # Bilan économique
    {"category": "Bilan économique", "question": "Comment les actifs sont-ils valorisés sous Solvabilité II ?", "expected_source_contains": ["actifs", "valorisation"]},
    {"category": "Bilan économique", "question": "Comment les passifs sont-ils valorisés sous Solvabilité II ?", "expected_source_contains": ["passifs", "valorisation"]},
    {"category": "Bilan économique", "question": "Quelle est la différence entre bilan comptable et bilan économique Solvabilité II ?", "expected_source_contains": ["bilan", "économique"]},
    {"category": "Bilan économique", "question": "Qu'est-ce que la juste valeur dans le bilan Solvabilité II ?", "expected_source_contains": ["juste valeur", "valorisation"]},
    {"category": "Bilan économique", "question": "Comment sont traités les impôts différés dans le bilan Solvabilité II ?", "expected_source_contains": ["impôts différés", "bilan"]},

    # Contrôle prudentiel
    {"category": "Contrôle prudentiel", "question": "Que se passe-t-il lorsqu'une entreprise ne couvre plus son SCR ?", "expected_source_contains": ["scr", "plan de rétablissement"]},
    {"category": "Contrôle prudentiel", "question": "Que se passe-t-il lorsqu'une entreprise ne couvre plus son MCR ?", "expected_source_contains": ["mcr", "plan de financement"]},
    {"category": "Contrôle prudentiel", "question": "Quels pouvoirs possède l'autorité de contrôle sous Solvabilité II ?", "expected_source_contains": ["autorité de contrôle", "pouvoirs"]},
    {"category": "Contrôle prudentiel", "question": "Qu'est-ce qu'un capital add-on ?", "expected_source_contains": ["capital add-on", "capital supplémentaire"]},
    {"category": "Contrôle prudentiel", "question": "Dans quels cas le superviseur peut-il imposer un capital add-on ?", "expected_source_contains": ["capital add-on", "superviseur"]},

    # Risques spécifiques
    {"category": "Risques spécifiques", "question": "Comment Solvabilité II traite-t-elle le risque de concentration ?", "expected_source_contains": ["concentration", "risque"]},
    {"category": "Risques spécifiques", "question": "Comment Solvabilité II traite-t-elle le risque de liquidité ?", "expected_source_contains": ["liquidité", "risque"]},
    {"category": "Risques spécifiques", "question": "Comment le risque de contrepartie est-il calculé dans la formule standard ?", "expected_source_contains": ["contrepartie", "formule standard"]},
    {"category": "Risques spécifiques", "question": "Qu'est-ce que le risque de défaut de contrepartie ?", "expected_source_contains": ["défaut", "contrepartie"]},
    {"category": "Risques spécifiques", "question": "Pourquoi les stress tests sont-ils importants dans l'ORSA ?", "expected_source_contains": ["stress test", "orsa"]},
    {"category": "Risques spécifiques", "question": "Comment les scénarios défavorables sont-ils utilisés dans l'ORSA ?", "expected_source_contains": ["scénarios", "orsa"]},

    # Proportionnalité
    {"category": "Proportionnalité", "question": "Comment le principe de proportionnalité s'applique-t-il aux petites entreprises d'assurance ?", "expected_source_contains": ["proportionnalité", "entreprise"]},
    {"category": "Proportionnalité", "question": "Qu'est-ce que la matérialité dans le contexte Solvabilité II ?", "expected_source_contains": ["matérialité", "risque"]},
    {"category": "Proportionnalité", "question": "Comment documenter une approche proportionnée sous Solvabilité II ?", "expected_source_contains": ["proportionnalité", "documentation"]},

    # Transversal
    {"category": "Transversal", "question": "Quels sont les trois piliers de Solvabilité II ?", "expected_source_contains": ["pilier", "exigences quantitatives", "gouvernance", "reporting"]},
    {"category": "Transversal", "question": "Quel est l'objectif général de Solvabilité II ?", "expected_source_contains": ["protection des assurés", "solvabilité"]},
    {"category": "Transversal", "question": "Comment Solvabilité II protège-t-elle les preneurs d'assurance ?", "expected_source_contains": ["preneurs d'assurance", "protection"]},
    {"category": "Transversal", "question": "Quelle est la logique économique de Solvabilité II ?", "expected_source_contains": ["économique", "risque"]},
    {"category": "Transversal", "question": "Comment Solvabilité II combine-t-elle exigences quantitatives et gouvernance ?", "expected_source_contains": ["exigences quantitatives", "gouvernance"]},
    {"category": "Transversal", "question": "Pourquoi la documentation est-elle importante sous Solvabilité II ?", "expected_source_contains": ["documentation", "gouvernance"]},
]






# ============================================================
# Text Normalization Helpers
# ============================================================

def normalize_text(text: str) -> str:
    """
    Normalize text for simple lexical matching:
    - lowercase
    - remove accents
    - normalize apostrophes
    - collapse whitespace
    """

    if text is None:
        return ""

    text = str(text).lower()
    text = text.replace("’", "'").replace("`", "'")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text


def term_matches(term: str, blob: str) -> bool:
    """
    Check whether a normalized expected term is present in a normalized chunk blob.
    """

    return normalize_text(term) in blob


# ============================================================
# Retrieval Evaluation Function
# ============================================================

def evaluate_retrieval(
    eval_questions: list[dict] | None = None,
    k: int = TOP_K_FINAL,
    min_coverage: float = 0.5,
    use_reranker: bool = False,
) -> pd.DataFrame:
    """
    Evaluate the retrieval quality of the Solvency II RAG pipeline.

    For each question:
    - retrieve the top-k chunks
    - check whether expected terms appear in each retrieved chunk
    - record hit@k, first matching rank, MRR and matched terms

    Parameters
    ----------
    eval_questions : list[dict] | None
        List of evaluation questions.
    k : int
        Number of chunks retrieved.
    min_coverage : float
        Minimum share of expected terms that must be found in a chunk
        for the retrieval to count as a hit.
    use_reranker : bool
        Whether to evaluate retrieval with reranking enabled.

    Returns
    -------
    pd.DataFrame
        Evaluation results by question.
    """

    if eval_questions is None:
        eval_questions = EVAL_QUESTIONS

    retriever = SolvencyRetriever(use_reranker=use_reranker)
    rows = []

    for item in eval_questions:
        category = item.get("category", "Unclassified")
        question = item["question"]

        expected_terms = item.get("expected_source_contains", [])
        normalized_expected_terms = [
            normalize_text(term)
            for term in expected_terms
        ]

        chunks = retriever.retrieve(question, k=k)

        first_hit_rank = None
        matched_terms_at_first_hit = []
        coverage_at_first_hit = 0.0

        for rank, chunk in enumerate(chunks, start=1):
            blob = normalize_text(
                f"{chunk.source_name} {chunk.section or ''} {chunk.text}"
            )

            matched_terms = [
                original_term
                for original_term, normalized_term in zip(
                    expected_terms,
                    normalized_expected_terms,
                )
                if normalized_term in blob
            ]

            coverage = (
                len(matched_terms) / len(expected_terms)
                if expected_terms
                else 1.0
            )

            if coverage >= min_coverage:
                first_hit_rank = rank
                matched_terms_at_first_hit = matched_terms
                coverage_at_first_hit = coverage
                break

        top_sources = "; ".join(
            format_citation(chunk, i + 1)
            for i, chunk in enumerate(chunks[:3])
        )

        rows.append({
            "category": category,
            "question": question,
            "expected_terms": expected_terms,
            "match_rule": f"term coverage >= {int(min_coverage * 100)}%",
            f"hit@{k}": first_hit_rank is not None,
            "mrr": 0.0 if first_hit_rank is None else round(1 / first_hit_rank, 3),
            "first_hit_rank": first_hit_rank,
            "coverage_at_first_hit": round(coverage_at_first_hit, 3),
            "matched_terms_at_first_hit": matched_terms_at_first_hit,
            "missing_terms": [
                term for term in expected_terms
                if term not in matched_terms_at_first_hit
            ],
            "top_sources": top_sources,
        })

    return pd.DataFrame(rows)


# ============================================================
# Summary Function
# Useful for LinkedIn / GitHub / README metrics
# ============================================================

def summarize_retrieval_evaluation(
    evaluation: pd.DataFrame,
    k: int = TOP_K_FINAL,
) -> pd.DataFrame:
    """
    Summarize retrieval performance by category.

    Parameters
    ----------
    evaluation : pd.DataFrame
        Output of evaluate_retrieval().
    k : int
        Same top-k value used during retrieval evaluation.

    Returns
    -------
    pd.DataFrame
        Summary metrics by category.
    """

    hit_col = f"hit@{k}"

    summary = (
        evaluation
        .groupby("category", dropna=False)
        .agg(
            questions=("question", "count"),
            hit_rate=(hit_col, "mean"),
            mean_mrr=("mrr", "mean"),
            mean_first_hit_rank=("first_hit_rank", "mean"),
            mean_coverage=("coverage_at_first_hit", "mean"),
        )
        .reset_index()
    )

    summary["hit_rate"] = summary["hit_rate"].round(3)
    summary["mean_mrr"] = summary["mean_mrr"].round(3)
    summary["mean_first_hit_rank"] = summary["mean_first_hit_rank"].round(2)
    summary["mean_coverage"] = summary["mean_coverage"].round(3)

    return summary.sort_values(
        by=["hit_rate", "mean_mrr"],
        ascending=[False, False],
    )


# ============================================================
# Global Summary Function
# ============================================================

def retrieval_global_metrics(
    evaluation: pd.DataFrame,
    k: int = TOP_K_FINAL,
) -> dict:
    """
    Return global retrieval metrics as a dictionary.
    """

    hit_col = f"hit@{k}"

    return {
        "questions": len(evaluation),
        f"hit@{k}": round(evaluation[hit_col].mean(), 3),
        "mean_mrr": round(evaluation["mrr"].mean(), 3),
        "mean_first_hit_rank": round(evaluation["first_hit_rank"].mean(), 2),
        "mean_coverage": round(evaluation["coverage_at_first_hit"].mean(), 3),
    }


# ============================================================
# Manual Run in Notebook
# ============================================================

evaluation = evaluate_retrieval(
     eval_questions=EVAL_QUESTIONS,
     k=TOP_K_FINAL,
     min_coverage=0.5,
     use_reranker=False,
)

# evaluation

summary = summarize_retrieval_evaluation(evaluation, k=TOP_K_FINAL)
summary

global_metrics = retrieval_global_metrics(evaluation, k=TOP_K_FINAL)
global_metrics


Chunks déjà à jour.
Indexed 128/9517 chunks
Indexed 256/9517 chunks
Indexed 384/9517 chunks
Indexed 512/9517 chunks
Indexed 640/9517 chunks
Indexed 768/9517 chunks
Indexed 896/9517 chunks
Indexed 1024/9517 chunks
Indexed 1152/9517 chunks
Indexed 1280/9517 chunks
Indexed 1408/9517 chunks
Indexed 1536/9517 chunks
Indexed 1664/9517 chunks
Indexed 1792/9517 chunks
Indexed 1920/9517 chunks
Indexed 2048/9517 chunks
Indexed 2176/9517 chunks
Indexed 2304/9517 chunks
Indexed 2432/9517 chunks
Indexed 2560/9517 chunks
Indexed 2688/9517 chunks
Indexed 2816/9517 chunks
Indexed 2944/9517 chunks
Indexed 3072/9517 chunks
Indexed 3200/9517 chunks
Indexed 3328/9517 chunks
Indexed 3456/9517 chunks
Indexed 3584/9517 chunks
Indexed 3712/9517 chunks
Indexed 3840/9517 chunks
Indexed 3968/9517 chunks
Indexed 4096/9517 chunks
Indexed 4224/9517 chunks
Indexed 4352/9517 chunks
Indexed 4480/9517 chunks
Indexed 4608/9517 chunks
Indexed 4736/9517 chunks
Indexed 4864/9517 chunks
Indexed 4992/9517 chunks
Indexed 5120

{'questions': 127,
 'hit@4': 0.898,
 'mean_mrr': 0.84,
 'mean_first_hit_rank': 1.16,
 'mean_coverage': 0.69}

In [17]:
def dataframe_to_markdown_simple(df: pd.DataFrame) -> str:
    """
    Convert a DataFrame to a simple Markdown table without requiring tabulate.
    """

    if df.empty:
        return "_No data available._"

    columns = list(df.columns)

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = []
    for _, row in df.iterrows():
        values = [str(row[col]) for col in columns]
        rows.append("| " + " | ".join(values) + " |")

    return "\n".join([header, separator] + rows)


def generate_retrieval_eval_doc_summary(
    evaluation: pd.DataFrame,
    summary: pd.DataFrame,
    k: int = TOP_K_FINAL,
) -> str:
    hit_col = f"hit@{k}"

    total_questions = len(evaluation)
    hit_rate = evaluation[hit_col].mean()
    mean_mrr = evaluation["mrr"].mean()
    mean_first_hit_rank = evaluation["first_hit_rank"].mean()
    mean_coverage = evaluation["coverage_at_first_hit"].mean()

    failed_questions = evaluation[~evaluation[hit_col]]

    best_categories = summary.sort_values(
        by=["hit_rate", "mean_mrr"],
        ascending=[False, False],
    ).head(3)

    weakest_categories = summary.sort_values(
        by=["hit_rate", "mean_mrr"],
        ascending=[True, True],
    ).head(3)

    strongest_table = dataframe_to_markdown_simple(best_categories)
    weakest_table = dataframe_to_markdown_simple(weakest_categories)

    text = f"""
## Retrieval Evaluation Summary

The retrieval component was evaluated on **{total_questions} Solvency II questions** covering Pillar 1, Pillar 2, Pillar 3, group supervision, investments, reinsurance, internal models, prudential supervision and transversal topics.

For each question, the retriever returned the top-{k} most relevant chunks. A retrieval was counted as successful when at least 50% of the expected reference terms were found in one of the retrieved chunks.

### Global Results

| Metric | Value |
|---|---:|
| Questions evaluated | {total_questions} |
| Hit@{k} | {hit_rate:.3f} |
| Mean MRR | {mean_mrr:.3f} |
| Mean first hit rank | {mean_first_hit_rank:.2f} |
| Mean term coverage | {mean_coverage:.3f} |
| Failed questions | {len(failed_questions)} |

### Strongest Categories

{strongest_table}

### Weakest Categories

{weakest_table}

### Interpretation

The retrieval pipeline achieved a Hit@{k} of **{hit_rate:.1%}**, meaning that the expected source content was retrieved within the top-{k} chunks for most evaluation questions. The mean first hit rank of **{mean_first_hit_rank:.2f}** indicates that successful matches are generally ranked very high, often in the first retrieved positions.

Remaining failures are useful for improving chunking, query formulation, synonym handling, and coverage of regulatory terminology.
""".strip()

    return text


In [18]:
doc_summary = generate_retrieval_eval_doc_summary(
    evaluation=evaluation,
    summary=summary,
    k=TOP_K_FINAL,
)

print(doc_summary)


## Retrieval Evaluation Summary

The retrieval component was evaluated on **127 Solvency II questions** covering Pillar 1, Pillar 2, Pillar 3, group supervision, investments, reinsurance, internal models, prudential supervision and transversal topics.

For each question, the retriever returned the top-4 most relevant chunks. A retrieval was counted as successful when at least 50% of the expected reference terms were found in one of the retrieved chunks.

### Global Results

| Metric | Value |
|---|---:|
| Questions evaluated | 127 |
| Hit@4 | 0.898 |
| Mean MRR | 0.840 |
| Mean first hit rank | 1.16 |
| Mean term coverage | 0.690 |
| Failed questions | 13 |

### Strongest Categories

| category | questions | hit_rate | mean_mrr | mean_first_hit_rank | mean_coverage |
| --- | --- | --- | --- | --- | --- |
| Pilier 1 - Fonds propres | 7 | 1.0 | 1.0 | 1.0 | 0.762 |
| Pilier 2 - ORSA | 4 | 1.0 | 1.0 | 1.0 | 0.625 |
| Réassurance | 5 | 1.0 | 1.0 | 1.0 | 0.8 |

### Weakest Categories

| cate

## Checklist de production

Avant d’utiliser ce système pour un vrai travail Solvabilité II :

- Utiliser uniquement des documents sources approuvés.
- Enregistrer les versions des documents et leurs dates d’entrée en vigueur.
- Ajouter une validation OCR pour les PDF scannés.
- Vérifier manuellement la qualité des chunks sur un échantillon.
- Construire un jeu d’évaluation de référence et suivre la précision de la recherche.
- Exiger des réponses citées avec leurs sources.
- Please have human expertise
- Journaliser les questions, les chunks récupérés, les réponses, les versions des modèles et les versions des sources.
- Réindexer lorsque les documents EIOPA, UE, du superviseur local ou les politiques internes changent.
